## 1. Đọc dữ liệu

In [ ]:
import pandas as pd
from pathlib import Path

# Sửa PROJECT_DIR nếu notebook không nằm trong thư mục project.
PROJECT_DIR = Path.cwd().resolve()

# Nếu notebook được chạy ở thư mục con, tìm thư mục project có 01_craw_data.
if not (PROJECT_DIR / "01_craw_data").is_dir():
    for parent in PROJECT_DIR.parents:
        if (parent / "01_craw_data").is_dir():
            PROJECT_DIR = parent
            break

DATA_DIR = PROJECT_DIR / "01_craw_data" 

category = pd.read_csv(DATA_DIR / 'category' / 'category.csv', encoding='utf-8')
product = pd.read_csv(DATA_DIR / 'product' / 'product.csv', encoding='utf-8')
product_detail = pd.read_csv(DATA_DIR / 'product' / 'product_detail.csv', encoding='utf-8')

product_pareto = DATA_DIR / "daily_price" / "product_pareto.csv"

print('Danh mục:', category.shape[0])
print('Số lượng sản phẩm:', product_detail.shape[0])

display(category.head())
display(product.head())
display(product_detail.head())

# 2. Chuẩn hóa dữ liệu và Merge product với product_detail

In [ ]:
product_detail = product_detail[
    ["product_id", "all_time_quantity_sold"]
]

product_detail.fillna(0, inplace=True)

df = product_detail.merge(
    product[["product_id", "category_id"]],
    on="product_id",
    how="left"
).merge(
    category[["category_id", "name"]],
    on="category_id",
    how="left"
).drop_duplicates()

display(df.head())
display(df.info())

# 3. Phân tích pareto danh mục theo all_time_quantity_sold

In [ ]:
df_pareto = (
    df.groupby(["category_id", "name"], as_index=False)
    .agg(total_quantity_sold=("all_time_quantity_sold", "sum"))
    .sort_values("total_quantity_sold", ascending=False)
    .reset_index()
)

# Tính tỷ trọng doanh số
total_sold = df_pareto["total_quantity_sold"].sum()

df_pareto["sales_share"] = (
    df_pareto["total_quantity_sold"] / total_sold * 100
)

# Tính tỷ trọng lũy kế
df_pareto["cumulative_share"] = df_pareto["sales_share"].cumsum()

# Phân nhóm Pareto: nhóm danh mục đóng góp đến 80%
df_pareto["pareto_group"] = (
    df_pareto["cumulative_share"] <= 80
).map({True: "Top 80%", False: "Remaining 20%"})

display(df_pareto)

# 4. Lấy các product_id thuộc danh mục top 80% để theo dõi giá hằng ngày

In [ ]:
# Lấy danh sách category_id thuộc nhóm Top 80%
top_categories = df_pareto.loc[
    df_pareto["pareto_group"] == "Top 80%",
    "category_id"
]

# Lấy danh sách product_id thuộc các danh mục Top 80%
df_product_top80 = (
    df.loc[df["category_id"].isin(top_categories), ["product_id"]]
    .drop_duplicates()
    .reset_index(drop=True)
)

# Merge với product để lấy đầy đủ thông tin
df_product_top80 = df_product_top80.merge(
    product[["product_id", "seller_product_id", "category_id"]],
    on="product_id",
    how="left"
)
display(df_product_top80.head())

## 5. Lọc bỏ các sản phẩm thuộc danh mục Voucher - Dịch vụ (category_id = 11312)

> **Lý do**: Voucher, thẻ cào và dịch vụ số không phải hàng hóa vật lý. Cơ chế giá và biến động giá của voucher không phản ánh đúng quy luật cung cầu thị trường bán lẻ. Việc đưa voucher vào theo dõi giá hằng ngày sẽ làm sai lệch các chỉ số phân tích giá (ASP, biên độ giảm giá, độ co giãn giá).

In [ ]:
# Lọc bỏ danh mục Voucher - Dịch vụ (category_id = 11312)
VOUCHER_CATEGORY_ID = 11312
initial_count = len(df_product_top80)
voucher_count = (df_product_top80['category_id'] == VOUCHER_CATEGORY_ID).sum()

df_product_top80 = df_product_top80.loc[
    df_product_top80['category_id'] != VOUCHER_CATEGORY_ID
].reset_index(drop=True)

print(f'Tổng số sản phẩm trước khi lọc: {initial_count:,}')
print(f'Số lượng sản phẩm Voucher (11312) đã loại bỏ: {voucher_count:,}')
print(f'Số lượng sản phẩm Pareto sạch còn lại để theo dõi giá: {len(df_product_top80):,}')
display(df_product_top80.head())

## 6. Lưu vào thư mục 01_craw_data/daily_price để crawl giá hằng ngày

In [ ]:
# Lưu danh sách sản phẩm Pareto vào 01_craw_data/daily_price/product_pareto.csv
# df_product_top80.to_csv(
#     product_pareto,
#     index=False,
#     encoding='utf-8-sig'
# )
print(f'Đã lưu thành công {len(df_product_top80):,} sản phẩm Pareto (không có voucher) tại:\n{product_pareto}')